<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/log2XYZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import re

# ============================================================
# PATHS
# ============================================================
log_folder = "/content/drive/MyDrive/log-files"
xyz_folder = "/content/drive/MyDrive/Suman_files"

# Check LOG folder — DO NOT create it
if not os.path.isdir(log_folder):
    raise FileNotFoundError(
        f"LOG folder not found: {log_folder}"
    )

# Create output folder only
os.makedirs(xyz_folder, exist_ok=True)

print("Google Drive mounted")
print("LOG folder:", log_folder)
print("XYZ folder:", xyz_folder)


# ============================================================
# ATOMIC NUMBER -> ELEMENT SYMBOL
# (full table, borrowed from extractOptimizedCoords.py — the
# original only went up to Xe/54, which silently produced "X"
# for any heavier element)
# ============================================================
atomic_symbol_str = {
    "1": "H", "2": "He", "3": "Li", "4": "Be", "5": "B",
    "6": "C", "7": "N", "8": "O", "9": "F", "10": "Ne",
    "11": "Na", "12": "Mg", "13": "Al", "14": "Si", "15": "P",
    "16": "S", "17": "Cl", "18": "Ar", "19": "K", "20": "Ca",
    "21": "Sc", "22": "Ti", "23": "V", "24": "Cr", "25": "Mn",
    "26": "Fe", "27": "Co", "28": "Ni", "29": "Cu", "30": "Zn",
    "31": "Ga", "32": "Ge", "33": "As", "34": "Se", "35": "Br",
    "36": "Kr", "37": "Rb", "38": "Sr", "39": "Y", "40": "Zr",
    "41": "Nb", "42": "Mo", "43": "Tc", "44": "Ru", "45": "Rh",
    "46": "Pd", "47": "Ag", "48": "Cd", "49": "In", "50": "Sn",
    "51": "Sb", "52": "Te", "53": "I", "54": "Xe", "55": "Cs",
    "56": "Ba", "57": "La", "58": "Ce", "59": "Pr", "60": "Nd",
    "61": "Pm", "62": "Sm", "63": "Eu", "64": "Gd", "65": "Tb",
    "66": "Dy", "67": "Ho", "68": "Er", "69": "Tm", "70": "Yb",
    "71": "Lu", "72": "Hf", "73": "Ta", "74": "W", "75": "Re",
    "76": "Os", "77": "Ir", "78": "Pt", "79": "Au", "80": "Hg",
    "81": "Tl", "82": "Pb", "83": "Bi", "84": "Po", "85": "At",
    "86": "Rn", "87": "Fr", "88": "Ra", "89": "Ac", "90": "Th",
    "91": "Pa", "92": "U", "93": "Np", "94": "Pu", "95": "Am",
    "96": "Cm", "97": "Bk", "98": "Cf", "99": "Es", "100": "Fm",
    "101": "Md", "102": "No", "103": "Lr", "104": "Rf", "105": "Db",
    "106": "Sg", "107": "Bh", "108": "Hs", "109": "Mt", "110": "Ds",
    "111": "Rg", "112": "Uub", "113": "Uut", "114": "Uuq", "115": "Uup",
    "116": "Uuh", "117": "Uus", "118": "Uuo",
}
atomic_symbol = {int(k): v for k, v in atomic_symbol_str.items()}


# ============================================================
# EXTRACT COORDINATE BLOCKS, EACH PAIRED WITH ITS SCF ENERGY
# ============================================================
def extract_coordinate_blocks_with_energy(lines):
    """
    Returns a list of dicts: {"atoms": [...], "energy": float or None}
    Each block is paired with the first "SCF Done" energy that appears
    *after* it (i.e. the energy computed for that geometry), the same
    way extractOptimizedCoords.py associates an energy with each
    structure block.
    """

    blocks = []
    i = 0

    while i < len(lines):

        if (
            "Standard orientation:" in lines[i]
            or "Input orientation:" in lines[i]
        ):

            # Gaussian prints TWO dashed lines with two header lines
            # between them before the actual coordinate rows begin:
            #   Input orientation:
            #    ---------------------------------------------------  <- dash 1
            #    Center     Atomic      Atomic     Coordinates ...    <- header 1
            #    Number     Number       Type       X   Y   Z         <- header 2
            #    ---------------------------------------------------  <- dash 2
            #         1         46           0    0.614482 ...        <- data starts here
            # We must skip past BOTH dashed lines, not just the first.
            start = i + 1

            # Find first dashed line
            while start < len(lines):
                if lines[start].strip().startswith("-----"):
                    break
                start += 1

            start += 1

            # Find second dashed line (skips the two header lines)
            while start < len(lines):
                if lines[start].strip().startswith("-----"):
                    break
                start += 1

            start += 1

            atoms = []

            while start < len(lines):

                line = lines[start].strip()

                # End of coordinate table
                if line.startswith("-----"):
                    break

                parts = line.split()

                if len(parts) >= 6:
                    try:
                        atomic_number = int(parts[1])
                        x = float(parts[3])
                        y = float(parts[4])
                        z = float(parts[5])
                        atoms.append((atomic_number, x, y, z))
                    except ValueError:
                        pass

                start += 1

            if atoms:
                # Look ahead for the SCF energy belonging to this geometry
                energy = None
                j = start
                while j < len(lines):
                    if "Standard orientation:" in lines[j] or "Input orientation:" in lines[j]:
                        # hit the next geometry block before finding an SCF energy
                        break
                    if "SCF Done:" in lines[j]:
                        match = re.search(r"SCF Done:.*?=\s*(-?\d+\.\d+)", lines[j])
                        if match:
                            try:
                                energy = float(match.group(1))
                            except ValueError:
                                energy = None
                        break
                    j += 1

                blocks.append({"atoms": atoms, "energy": energy})

            i = start

        i += 1

    return blocks


# ============================================================
# CHOOSE THE RIGHT STRUCTURE
# (mirrors extractOptimizedCoords.py: use the converged geometry
# if the job optimized successfully; otherwise fall back to the
# structure with the lowest SCF energy, instead of just grabbing
# whatever the last block in the file happens to be)
# ============================================================
def select_structure(lines, blocks):

    optimized = any("Optimization completed" in line for line in lines)

    if optimized:
        return blocks[-1]["atoms"], True

    # Fallback: lowest-energy structure. Missing energies get a very
    # high placeholder (1000.0) so they sort last, same as
    # extractOptimizedCoords.py's getEnergy() default.
    best = min(blocks, key=lambda b: b["energy"] if b["energy"] is not None else 1000.0)
    return best["atoms"], False


# ============================================================
# CONVERT LOG -> XYZ
# ============================================================
def convert_log_to_xyz(log_path, xyz_path):

    with open(log_path, "r", errors="ignore") as f:
        lines = f.readlines()

    coordinate_blocks = extract_coordinate_blocks_with_energy(lines)

    if not coordinate_blocks:
        print(f"❌ {os.path.basename(log_path)}: No coordinates found")
        return False

    atoms, optimized = select_structure(lines, coordinate_blocks)

    name = os.path.splitext(os.path.basename(log_path))[0]

    with open(xyz_path, "w") as fout:
        fout.write(f"{len(atoms)}\n")
        fout.write(f"{name}\n")

        for atomic_number, x, y, z in atoms:
            symbol = atomic_symbol.get(atomic_number, "X")
            fout.write(f"{symbol:2s} {x:14.8f} {y:14.8f} {z:14.8f}\n")

    status = "optimized" if optimized else "lowest-energy fallback"
    print(f"✅ Converted: {name}.log → {name}.xyz  ({status})")
    print(f"   Atoms: {len(atoms)}")
    print(f"   Coordinate blocks: {len(coordinate_blocks)}")

    return True


# ============================================================
# CONVERT ALL LOG FILES
# ============================================================
print("\n" + "=" * 70)
print("CONVERTING LOG → XYZ")
print("=" * 70)

log_files = sorted(
    f for f in os.listdir(log_folder) if f.lower().endswith(".log")
)

print(f"\nFound {len(log_files)} LOG files\n")

converted = 0
failed = 0

for file in log_files:
    log_path = os.path.join(log_folder, file)
    xyz_name = os.path.splitext(file)[0] + ".xyz"
    xyz_path = os.path.join(xyz_folder, xyz_name)

    if convert_log_to_xyz(log_path, xyz_path):
        converted += 1
    else:
        failed += 1


# ============================================================
# CREATE SI.txt
# ============================================================
print("\n" + "=" * 70)
print("CREATING SI.txt")
print("=" * 70)

si_file = os.path.join(xyz_folder, "SI.txt")

xyz_files = sorted(
    f for f in os.listdir(xyz_folder) if f.lower().endswith(".xyz")
)

with open(si_file, "w", errors="ignore") as fout:
    for file in xyz_files:
        xyz_path = os.path.join(xyz_folder, file)
        with open(xyz_path, "r", errors="ignore") as fin:
            content = fin.read()

        name = os.path.splitext(file)[0]

        fout.write("=" * 70 + "\n")
        fout.write(name + "\n")
        fout.write("=" * 70 + "\n")
        fout.write(content)
        fout.write("\n\n")


# ============================================================
# FINAL REPORT
# ============================================================
print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print(f"Converted : {converted}")
print(f"Failed    : {failed}")
print(f"XYZ files : {xyz_folder}")
print(f"SI file   : {si_file}")
print("\nXYZ files created:")
for file in xyz_files:
    print("   ", file)

Mounted at /content/drive
Google Drive mounted
LOG folder: /content/drive/MyDrive/log-files
XYZ folder: /content/drive/MyDrive/Suman_files

CONVERTING LOG → XYZ

Found 19 LOG files

✅ Converted: L10_new.log → L10_new.xyz  (optimized)
   Atoms: 37
   Coordinate blocks: 6
✅ Converted: L11_new.log → L11_new.xyz  (optimized)
   Atoms: 41
   Coordinate blocks: 12
✅ Converted: L12_new.log → L12_new.xyz  (optimized)
   Atoms: 43
   Coordinate blocks: 12
✅ Converted: L13_new.log → L13_new.xyz  (optimized)
   Atoms: 43
   Coordinate blocks: 16
✅ Converted: L14-new.log → L14-new.xyz  (optimized)
   Atoms: 48
   Coordinate blocks: 12
✅ Converted: L15_new.log → L15_new.xyz  (optimized)
   Atoms: 38
   Coordinate blocks: 6
✅ Converted: L16_new.log → L16_new.xyz  (optimized)
   Atoms: 41
   Coordinate blocks: 10
✅ Converted: L17_new.log → L17_new.xyz  (optimized)
   Atoms: 43
   Coordinate blocks: 12
✅ Converted: L18_new.log → L18_new.xyz  (optimized)
   Atoms: 46
   Coordinate blocks: 10
✅ Converte